In [1]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from util.rescale_to_bangalore import rescale_to_city_bbox
from sklearn.ensemble import RandomForestRegressor as RFR
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from util.process_geohash import GeohashProcessor
from lightgbm import LGBMRegressor as lgbmr
from collections import defaultdict
import lightgbm as lgb
import pandas as pd
import numpy as np
import pickle

In [2]:
import sys, os

In [3]:
sys.path.append(os.path.abspath("..."))

In [52]:
df = pd.read_csv("dataset/train.csv")
print(df.isna().sum())
print("\n")
print(df["Weather"].value_counts())
print("\n")
print(df["RoadType"].value_counts())
print("\n")
print("total_timestamps", df["timestamp"].value_counts().size)

Index               0
geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
dtype: int64


Weather
Sunny    27717
Rainy    20824
Foggy    20243
Snowy     7718
Name: count, dtype: int64


RoadType
Residential    69230
Street          3909
Highway         3560
Name: count, dtype: int64


total_timestamps 96


In [26]:
df.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [27]:
dftest = pd.read_csv("lag.csv")
dftest.columns

Index(['Unnamed: 0', 'Index', 'geohash', 'day', 'demand', 'NumberofLanes',
       'LargeVehicles', 'Landmarks', 'Temperature', 'time_minutes', 'hour',
       'sin_hour', 'cos_hour', 'sin_minute', 'cos_minute', 'is_rush_hour',
       'night_time', 'lat', 'lon', 'geo_zone', 'dist_to_center', 'geo_cluster',
       'demand_lag_15', 'demand_lag_hour', 'demand_lag_day',
       'demand_roll_mean_hour', 'demand_roll_std_hour', 'RoadType_Highway',
       'RoadType_Residential', 'RoadType_Street', 'RoadType_nan',
       'Weather_Foggy', 'Weather_Rainy', 'Weather_Snowy', 'Weather_Sunny',
       'Weather_nan'],
      dtype='str')

In [28]:
lst1 = list(df["RoadType"].unique())
lst2 = list(df["Weather"].unique())
print(lst1, lst2)

[nan, 'Residential', 'Street', 'Highway'] [nan, 'Sunny', 'Rainy', 'Foggy', 'Snowy']


#Temporal and geohash preprocessing

In [29]:
def time_change(df):
  dfcopy = df.copy()
  dfcopy["time_minutes"] = pd.to_datetime(dfcopy["timestamp"], format="%H:%M").dt.hour*60 + pd.to_datetime(dfcopy["timestamp"], format="%H:%M").dt.minute
  dfcopy["hour"] = dfcopy["time_minutes"]/60
  dfcopy["sin_hour"] = np.sin(2 * np.pi * dfcopy["hour"]/24)
  dfcopy["cos_hour"] = np.cos(2 * np.pi * dfcopy["hour"]/24)
  dfcopy["sin_minute"] = np.sin(2 * np.pi * dfcopy["time_minutes"]/1440)
  dfcopy["cos_minute"] = np.cos(2 * np.pi * dfcopy["time_minutes"]/1440)
  dfcopy["is_rush_hour"] = dfcopy["hour"].isin([8, 9, 10]).astype(int)
  dfcopy["night_time"] = dfcopy["hour"].isin([23, 1, 2, 4, ]).astype(int)

  return dfcopy

In [30]:
def labelEncode(df):
  df_copy = df.copy()

  df_copy["Temperature"] = df_copy["Temperature"].fillna(np.mean(df_copy["Temperature"]))
  df_copy["NumberofLanes"] = df_copy["NumberofLanes"].fillna(df_copy["NumberofLanes"].median())

  df_copy = df_copy.drop("timestamp", axis = 1)
  return df_copy

In [ ]:
def preprocess(df, le_geohash=None, le_large_vehicles=None, le_landmarks=None, oh_roadtype=None, oh_weather=None, fit_encoders=True):
    df_copy = df.copy()

    if fit_encoders:
        le_geohash = LabelEncoder()
        le_large_vehicles = LabelEncoder()
        le_landmarks = LabelEncoder()
        oh_roadtype = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        oh_weather = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

        df_copy["LargeVehicles"] = le_large_vehicles.fit_transform(df_copy["LargeVehicles"])
        df_copy["Landmarks"] = le_landmarks.fit_transform(df_copy["Landmarks"])
        df_copy["geohash"] = le_geohash.fit_transform()

        encoded = oh_roadtype.fit_transform(df_copy[["RoadType"]])
        encoded_w = oh_weather.fit_transform(df_copy[["Weather"]])
    else:
        if not all([le_large_vehicles, le_landmarks, oh_roadtype, oh_weather]):
            raise ValueError("Encoders must be provided when fit_encoders is False.")

        df_copy["LargeVehicles"] = le_large_vehicles.transform(df_copy["LargeVehicles"])
        df_copy["Landmarks"] = le_landmarks.transform(df_copy["Landmarks"])


        encoded = oh_roadtype.transform(df_copy[["RoadType"]])
        encoded_w = oh_weather.transform(df_copy[["Weather"]])

    encoded_arr = pd.DataFrame(encoded, columns=oh_roadtype.get_feature_names_out(["RoadType"]), index=df_copy.index)
    encoded_w_arr = pd.DataFrame(encoded_w, columns=oh_weather.get_feature_names_out(["Weather"]), index=df_copy.index)

    df_copy = df_copy.drop(["Weather", "RoadType"], axis=1)
    df_copy = pd.concat([df_copy, encoded_arr, encoded_w_arr], axis=1)

    return df_copy, le_large_vehicles, le_landmarks, oh_roadtype, oh_weather

In [32]:
def Lag(dfn, df):
  dfncopy = dfn.copy()
  dfncopy = dfn.sort_values(["geohash", "day", "timestamp"])
  dfncopy["demand_lag_15"] = df.groupby("geohash")["demand"].shift(1)
  dfncopy["demand_lag_hour"] = df.groupby("geohash")["demand"].shift(4)
  dfncopy["demand_lag_day"] = df.groupby("geohash")["demand"].shift(96)
  dfncopy["demand_roll_mean_hour"] = df.groupby("geohash")["demand"].shift(1).rolling(4).mean()
  dfncopy["demand_roll_std_hour"] = df.groupby("geohash")["demand"].shift(1).rolling(4).std()

  return dfncopy

#Implementing the functions

In [33]:
dfn = time_change(df)
gp = GeohashProcessor(n_clusters=15, zone_prefix_len=5)
dfn = gp.fit_transform(dfn)
dfn.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,...,cos_hour,sin_minute,cos_minute,is_rush_hour,night_time,lat,lon,geo_zone,dist_to_center,geo_cluster
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,...,1.0,0.0,1.0,0,0,-5.484924,90.664673,qp02z,0.168925,1
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,...,1.0,0.0,1.0,0,0,-5.462952,90.686646,qp02z,0.138313,1
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,...,1.0,0.0,1.0,0,0,-5.462952,90.708618,qp08b,0.127315,1
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,...,1.0,0.0,1.0,0,0,-5.462952,90.862427,qp08g,0.150985,10
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,...,1.0,0.0,1.0,0,0,-5.457458,90.675659,qp02z,0.140444,1


In [53]:
le_geohash = LabelEncoder()
le_large_vehicles = LabelEncoder()
le_landmarks = LabelEncoder()
oh_roadtype = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
oh_weather = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

df_copy = df.copy()
df_copy["LargeVehicles"] = le_large_vehicles.fit_transform(df_copy["LargeVehicles"])
df_copy["Landmarks"] = le_landmarks.fit_transform(df_copy["Landmarks"])
df_copy["geohash"] = le_geohash.fit_transform(df_copy["geohash"])
encoded = oh_roadtype.fit_transform(df_copy[["RoadType"]])
encoded_w = oh_weather.fit_transform(df_copy[["Weather"]])


In [54]:
oh_roadtype.categories_

[array(['Highway', 'Residential', 'Street', nan], dtype=object)]

In [55]:
df_copy.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,4,48,0:0,0.048804,NaN,1,1,0,NaN,NaN
1,1,25,48,0:0,0.118507,Residential,3,0,1,31.104565,Sunny
2,2,370,48,0:0,0.027132,Residential,1,1,0,25.919267,Sunny
3,3,416,48,0:0,0.003272,Residential,1,1,0,NaN,Rainy
4,4,22,48,0:0,0.010819,Residential,1,1,0,10.803667,Rainy


In [ ]:
with open("model/le_geohash.pkl", "wb") as f:
    pickle.dump(le_geohash, f)
f.close()

with open("model/le_landmarks.pkl", "wb") as f:
    pickle.dump(le_landmarks, f)
f.close()

with open("model/le_large_vehicles.pkl", "wb") as f:
    pickle.dump(le_large_vehicles, f)
f.close()

with open("model/le_roadtype.pkl", "wb") as f:
    pickle.dump(oh_roadtype, f)
f.close()

with open("model/le_weather.pkl", "wb") as f:
    pickle.dump(oh_weather, f)
f.close()

In [38]:
gp.save("geohashsaver.pkl")

In [13]:
dfn["geohash"] = le_geohash.fit_transform(dfn["geohash"])

In [14]:
dfn.columns

Index(['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType',
       'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather',
       'time_minutes', 'hour', 'sin_hour', 'cos_hour', 'sin_minute',
       'cos_minute', 'is_rush_hour', 'night_time', 'lat', 'lon', 'geo_zone',
       'dist_to_center', 'geo_cluster'],
      dtype='str')

#Lag and Rolling window

In [34]:
dfn = Lag(dfn, df)

In [35]:
dfn.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,...,lat,lon,geo_zone,dist_to_center,geo_cluster,demand_lag_15,demand_lag_hour,demand_lag_day,demand_roll_mean_hour,demand_roll_std_hour
36776,36776,qp02yc,48,10:30,0.046790,Residential,1,Not Allowed,No,22.809028,...,-5.484924,90.653687,qp02y,0.175617,1,0.017362,0.004853,NaN,0.019159,0.007330
37683,37683,qp02yc,48,10:45,0.021158,Residential,1,Not Allowed,No,16.543659,...,-5.484924,90.653687,qp02y,0.175617,1,0.046790,0.027092,NaN,0.012935,0.022599
2911,2911,qp02yc,48,1:0,0.005397,Residential,3,Allowed,Yes,11.299877,...,-5.484924,90.653687,qp02y,0.175617,1,NaN,NaN,NaN,NaN,NaN
8010,8010,qp02yc,48,2:30,0.012944,Residential,2,Not Allowed,Yes,32.142120,...,-5.484924,90.653687,qp02y,0.175617,1,0.005397,NaN,NaN,0.032407,0.019448
8905,8905,qp02yc,48,2:45,0.025961,Residential,1,Not Allowed,No,15.811063,...,-5.484924,90.653687,qp02y,0.175617,1,0.012944,NaN,NaN,0.040204,0.021864


In [36]:
dfn.to_csv("LagShift.csv")

In [17]:
dfn = labelEncode(dfn)
dfn, le_large_vehicles, le_landmarks, oh_roadtype, oh_weather = preprocess(dfn)

In [18]:
dfn.to_csv("lag.csv")

In [19]:
dfn = rescale_to_city_bbox(dfn)

In [20]:
dfn.head()

,Index,geohash,day,demand,NumberofLanes,LargeVehicles,Landmarks,Temperature,time_minutes,hour,...,RoadType_Residential,RoadType_Street,RoadType_nan,Weather_Foggy,Weather_Rainy,Weather_Snowy,Weather_Sunny,Weather_nan,display_lat,display_lon
36776,36776,0,48,0.046790,1,1,0,22.809028,630,10.50,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,12.846786,77.423714
37683,37683,0,48,0.021158,1,1,0,16.543659,645,10.75,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,12.846786,77.423714
2911,2911,0,48,0.005397,3,0,1,11.299877,60,1.00,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,12.846786,77.423714
8010,8010,0,48,0.012944,2,1,1,32.142120,150,2.50,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,12.846786,77.423714
8905,8905,0,48,0.025961,1,1,0,15.811063,165,2.75,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,12.846786,77.423714


In [21]:
dfn.columns

Index(['Index', 'geohash', 'day', 'demand', 'NumberofLanes', 'LargeVehicles',
       'Landmarks', 'Temperature', 'time_minutes', 'hour', 'sin_hour',
       'cos_hour', 'sin_minute', 'cos_minute', 'is_rush_hour', 'night_time',
       'lat', 'lon', 'geo_zone', 'dist_to_center', 'geo_cluster',
       'demand_lag_15', 'demand_lag_hour', 'demand_lag_day',
       'demand_roll_mean_hour', 'demand_roll_std_hour', 'RoadType_Highway',
       'RoadType_Residential', 'RoadType_Street', 'RoadType_nan',
       'Weather_Foggy', 'Weather_Rainy', 'Weather_Snowy', 'Weather_Sunny',
       'Weather_nan', 'display_lat', 'display_lon'],
      dtype='str')

#Training and Modeling

In [ ]:
['Index','geohash', 'day', 'NumberofLanes', 'LargeVehicles', 'Landmarks',
       'Temprature', 'time_minutes', 'hour', 'sin_hour', 'cos_hour',
       'sin_minute', 'cos_minute', 'is_rush_hour', 'night_time', 'lat', 'lon',
       'geo_zone', 'dist_to_center', 'geo_cluster', 'RoadType_Residential',
       'Weather_Sunny', 'demand_lag_15', 'demand_lag_hour', 'demand_lag_day',
       'demand_roll_mean_hour', 'demand_roll_std_hour']

['Index', 'geohash', 'day', 'demand', 'NumberofLanes', 'LargeVehicles',
       'Landmarks', 'Temperature', 'time_minutes', 'hour', 'sin_hour',
       'cos_hour', 'sin_minute', 'cos_minute', 'is_rush_hour', 'night_time',
       'lat', 'lon', 'geo_zone', 'dist_to_center', 'geo_cluster',
       'demand_lag_15', 'demand_lag_hour', 'demand_lag_day',
       'demand_roll_mean_hour', 'demand_roll_std_hour', 'RoadType_Highway',
       'RoadType_Residential', 'RoadType_Street', 'RoadType_nan',
       'Weather_Foggy', 'Weather_Rainy', 'Weather_Snowy', 'Weather_Sunny',
       'Weather_nan']


In [38]:
Y = dfn["demand"]
X = dfn.drop(["demand", "display_lat", "display_lon"],  axis = 1)

Xtrain, Xtest, Ytrain, Ytest = train_test_split(X, Y, test_size = 0.2, random_state = 2)

print(Xtrain.shape)
print(Ytrain.shape)
print(Xtest.shape)
print(Ytest.shape)

(61839, 34)
(61839,)
(15460, 34)
(15460,)


In [39]:
type(Ytest)

pandas.Series

In [40]:
model = lgbmr(
    objective = "regression",
    metric = "rsme",
    boosting_type = "gbdt",
    max_depth = 5,
    min_data_in_leaf = 10,
    n_estimator = 500,
    random_state = 42,
    verbose = -1,
    num_leaves = 16,
    bagging_fraction = 0.4,
    feature_fraction = 0.6
)

model.fit(Xtrain, Ytrain)

,num_leaves,16
,max_depth,5
,objective,'regression'
,random_state,42
,metric,'rsme'
,min_data_in_leaf,10
,n_estimator,500
,verbose,-1
,bagging_fraction,0.4
,feature_fraction,0.6
,boosting_type,'gbdt'


In [41]:
Ypred = model.predict(Xtest)
r2 = r2_score(Ytest, Ypred)
print("r2_score", r2)

r2_score 0.9614152988690531


In [ ]:
filename = "traffic_model.pkl"

with open(filename, "wb")as f:
  pickle.dump(model, f)
f.close()

In [43]:
labelfile = "labelEnc.pkl"
with open(labelfile, "wb") as l:
    pickle.dump(le_geohash, l)
f.close()